In [1]:
from huggingface_hub import snapshot_download
import subprocess, sys
from pathlib import Path

MODELS_DIR = Path("/kaggle/working/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DOWNLOADS = [
    ("Qwen/Qwen3.5-9B",        "qwen_35_9b"),
]

# Set HF_TOKEN if models require authentication (gemma needs it)
import os
HF_TOKEN = os.environ.get("HF_TOKEN")  # set in Kaggle Secrets
if HF_TOKEN:
    print("  HF_TOKEN: set")
else:
    print("  WARNING: HF_TOKEN not set — gemma-4 download will fail (gated model)")
    print("  Add HF_TOKEN in: Kaggle → Add-ons → Secrets")

for repo_id, local_name in MODEL_DOWNLOADS:
    local_dir = MODELS_DIR / local_name
    if local_dir.exists() and any(local_dir.glob("*.safetensors")):
        print(f"  ✓ {local_name} already downloaded — skipping")
        continue
    print(f"\n▶ Downloading {repo_id} → {local_dir} ...")
    snapshot_download(
        repo_id=repo_id,
        local_dir=str(local_dir),
        token=HF_TOKEN,
        ignore_patterns=["*.pt", "original/*"],  # skip pytorch & original checkpoints
    )
    safetensors = list(local_dir.glob("*.safetensors"))
    print(f"  ✓ {local_name} done — {len(safetensors)} safetensors files")

  Add HF_TOKEN in: Kaggle → Add-ons → Secrets

▶ Downloading Qwen/Qwen3.5-9B → /kaggle/working/models/qwen_35_9b ...


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

  ✓ qwen_35_9b done — 4 safetensors files


In [2]:
import os

print("▶ Disk usage summary:")
result = subprocess.run(["du", "-sh", str(MODELS_DIR)],
                        capture_output=True, text=True)
print(result.stdout)

print("\n▶ Model directory contents:")
for model_dir in sorted(MODELS_DIR.iterdir()):
    files = list(model_dir.iterdir())
    size_mb = sum(f.stat().st_size for f in files if f.is_file()) / 1024**2
    print(f"  {model_dir.name}/  ({len(files)} files, {size_mb:.0f} MB)")
    for f in sorted(files)[:5]:
        print(f"    {f.name}")
    if len(files) > 5:
        print(f"    ... ({len(files)} total)")

print("\n✓ Setup complete. Commit this notebook to save outputs as Kaggle dataset.")

▶ Disk usage summary:
19G	/kaggle/working/models


▶ Model directory contents:
  qwen_35_9b/  (17 files, 18434 MB)
    .cache
    .gitattributes
    LICENSE
    README.md
    chat_template.jinja
    ... (17 total)

✓ Setup complete. Commit this notebook to save outputs as Kaggle dataset.
